# Step 1: Problem Understanding & Framing
## Business Case
In industrial settings, machinery operates under varying conditions, leading to different "states" (normal, high-wear, near-failure, etc.). Traditional predictive maintenance often relies on labeled failure data, which is expensive, rare, and imbalanced. The goal is to identify distinct operating states and anomalies in machinery telemetry without using failure labels during training. This enables early detection of risky behaviors, proactive maintenance, and optimization of schedules. This could result to reducing unplanned downtime and costs.

## Machine Learning Task
Unsupervised clustering can help group machine operational behaviors based on sensor telemetry (temperatures, speed, torque, tool wear, etc.). We will be able to see operating patters when we plot these clusters. Anything that falls out of those patterns, or forms a tiny cluster in its own, is often a sign of abnormal behavior or a developing issue. This is a typical approach in predictive maintenance to spot problems early.

## Technical Evaluation Metrics

* **Silhouette Score**: Measures how similar an instance is to its own cluster vs. others (range: -1 to 1; higher is better). Assesses cohesion and separation.
* **Davies-Bouldin Index**: Ratio of within-cluster scatter to between-cluster separation (lower is better).
* **Elbow Method (Inertia / Within-Cluster Sum of Squares - WCSS)**: Plots inertia vs. number of clusters (k) to find the "elbow" where additional clusters yield diminishing returns.

Additional diagnostics: Cluster stability, visualization of projections (PCA/t-SNE/UMAP), and post-hoc validation against held-out failure labels.

## Business KPIs

1. **Reduction in Unplanned Downtime**: Percentage decrease in unexpected failures by flagging "at-risk" clusters early (target: 15-30% reduction via timely interventions).
2. **Maintenance Schedule Optimization**: Increase in predictive (vs. reactive) maintenance actions and reduction in unnecessary preventive replacements (e.g., tool changes based on cluster-derived wear states).
3. **Overall Equipment Effectiveness (OEE) Improvement**: Measured via availability, performance, and quality uplift from better state awareness.

These KPIs tie directly to ROI through lower costs and higher throughput.

# Step 2: Data Collection & Understanding
## Dataset Overview
The AI4I 2020 Predictive Maintenance Dataset is a synthetic dataset (modeled on real industrial milling processes) with 10,000 instances and 14 columns. It includes process parameters and failure indicators. For strict unsupervised clustering, we will drop or separate all target/failure columns (Machine failure, TWF, HDF, PWF, OSF, RNF) during modeling. These will only be used post-hoc for cluster validation/profiling.
Key telemetry features capture machine operating conditions under different product quality variants.

Source: https://www.kaggle.com/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020

## Data Dictionary
| Feature            | Type        | Unit | Description / Operational Range                     | Role              |
|--------------------|-------------|------|------------------------------------------------------|-------------------|
| UDI                | Integer     | -    | Unique identifier (1–10000)                          | Drop (ID)         |
| Product ID         | Categorical | -    | L/M/H + serial number                                | Drop (ID)         |
| Type               | Categorical | -    | Product quality variant: L (50%), M (30%), H (20%)   | Feature (encode)  |
| Air temperature    | Continuous  | K    | ~300K ± 2K (random walk)                             | Feature           |
| Process temperature| Continuous  | K    | Air temp + ~10K ± 1K                                 | Feature           |
| Rotational speed   | Integer     | rpm  | Derived from ~2860W power, with noise                | Feature           |
| Torque             | Continuous  | Nm   | ~40 Nm ± 10 Nm (no negatives)                        | Feature           |
| Tool wear          | Integer     | min  | Cumulative wear; varies by Type (H/M/L add extra wear)| Feature          |
| Machine failure    | Binary      | -    | 0/1 (any failure)                                    | Drop for training |
| TWF / HDF / PWF / OSF / RNF | Binary | - | Specific failure modes                               | Drop for training |

## Data Undersrtanding

| Feature | Meaning | Why it matters for failure |
|---|---|---|
| Air temperature [K] | Ambient temp | High temp → overheating risk |
| Process temperature [K] | Internal temp | Strong indicator of thermal stress |
| Rotational speed [rpm] | Motor speed | Abnormal speed → mechanical wear |
| Torque [Nm] | Load on machine | High torque → strain → failure |
| Tool wear [min] | Usage time | Direct proxy for degradation |
| Type | Product type | Different products → different stress patterns |

**Notes:**
* Total ~3.4% failure rate (highly imbalanced, as expected in real PM data).
* No missing values.
* Features have different scales/magnitudes → scaling is critical for distance-based clustering.


# Step 3: Data Preprocessing, Applied EDA & Feature Engineering
## 3.1 Data Cleaning

* Remove non-predictive attributes (UDI, Product ID).
* Separate failure labels (Machine failure and failure mode columns). They are not needed in the unsupervised model training. They will be used for validation only.
* Check for missing values/duplicates (dataset is clean, but code includes guards).

* Handle the categorical values in the 'Type' column (L/M/H).

**Null values and Column Types**
![alt text](../presentation/images/eda_nulls_and_coltypes.png)

**Variable Summary**
![alt text](../presentation/images/eda_variable_summary.png)

**Selected Features Distribution**
![alt text](../presentation/images/eda_selected_feature_dist.png)

## 3.2 Applied EDA

* PCA
![alt text](../presentation/images/eda_clustering_tendency_hopkins.png)

  * Most points are concentrated in the middle. Hence, the machine states looks similar and failures may not be easily separable.  There's some outward spread but no distinct grouping. This likely tells us that there are no natural groupings (e.g., normal, warning, critical) that are separable.

* Domain context: Temperatures are correlated; power-related features (speed × torque) may be informative.

## 3.3 Feature Engineering & Scaling

* Scaling: Mandatory for K-Means, DBSCAN, Hierarchical (Euclidean distance). Use StandardScaler (robust to outliers in telemetry).
* Encoding: One-hot or ordinal for Type (quality affects wear).
* Derived Features:
  * temp_diff = Process temperature - Air temperature (heat dissipation proxy).
  * power_proxy = Torque * Rotational speed (approximates mechanical power/load).

* Feature Selection: Remove highly correlated features or low-variance ones to reduce noise.

## 3.4 Dimensionality Reduction

* PCA: Reduces dimensions while preserving variance; improves distance-based clustering and mitigates curse of dimensionality.
* t-SNE / UMAP: Visualization only (not for modeling) to inspect cluster separability in 2D.
![alt text](../presentation/images/eda_tsne.png)

Interpretation: 
**Failure Lablel**: 
* There is an absence of a distinct failure zone. Failure points overlaps heavily with normal operation.
* Machine Failures occur within the same operating space as normally operating machines.
* Unsupervised clustering alone is not enough to detect failures. Supervised learning is also needed. 

**Product Tye**
* Product types exhibit mild operational differences, but their t‑SNE projections show overlap, indicating that product type alone does not define distinct machine states

**Torque**
* Shows machine states into distinct regions, suggesting that torque is a key contributor to operational differences and potentially to failure risk i.e., High torque = high load = mechanical stress
* Stress increases wear and failure probability

## Step 4: Model Implementation & Comparison

We are going to implement three distinct clustering algorithms using the preprocessed features from Step 3. All code uses random_state=42. We evaluate using Silhouette Score, Davies-Bouldin Index, and Elbow Method (Inertia).

### Model Comparison
| Model                | Silhouette Score | Davies-Bouldin Index | Notes                               |
|----------------------|------------------|-----------------------|--------------------------------------|
| K-Means (k=3)        | 0.2334           | 1.3628                | Interpretable centroids              |
| DBSCAN               | -0.0562          | 2.5442                | Good for anomaly detection (noise)   |
| Hierarchical (k=4)   | 0.1655           | 1.6724                | Visual hierarchy                     |

![alt text](../presentation/images/clustering_elbow_silhoutte.png)

![alt text](../presentation/images/clustering_hierarchical.png)


Based on the elbow method (k ≈ 3–4) and silhouette analysis (peak at k = 2–3), we select **k = 3** as the optimal number of clusters. This choice balances compactness and separation while producing interpretable operating regimes relevant to predictive maintenance.

**For this case, our champion model is K-Mean(k-=3)**



## Step 5: Critical Thinking, Ethical AI & Bias Auditing

### Key Limitations of This Unsupervised Approach:

* No direct optimization against a business metric (e.g., failure prediction) during training — relies on post-hoc validation.
* Clusters may be unstable with new data distributions (concept drift in real factories).
* High-dimensional telemetry can still suffer from curse of dimensionality even after PCA.
* Cannot distinguish rare but critical failures if they don't form dense clusters.
* Requires domain expert interpretation of clusters.

**Mitigation in Production**: Combine with supervised models on labeled data when available, use online clustering for drift detection, and implement human-in-the-loop review.

### Bias & Fairness Audit
**Focus**: Product quality variant (Type: L, M, H) is a sensitive attribute (low-quality machines may be cheaper and more prone to issues).

Proportion of each Type in Clusters:
<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>cluster</th>
      <th>0</th>
      <th>1</th>
      <th>2</th>
      <th>3</th>
    </tr>
    <tr>
      <th>Type</th>
      <th></th>
      <th></th>
      <th></th>
      <th></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>H</th>
      <td>0.396</td>
      <td>0.165</td>
      <td>0.216</td>
      <td>0.223</td>
    </tr>
    <tr>
      <th>L</th>
      <td>0.349</td>
      <td>0.175</td>
      <td>0.234</td>
      <td>0.242</td>
    </tr>
    <tr>
      <th>M</th>
      <td>0.351</td>
      <td>0.169</td>
      <td>0.250</td>
      <td>0.230</td>
    </tr>
  </tbody>
</table>
</div>

Failure Rate by Type & Cluster:
<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th></th>
      <th>mean</th>
      <th>count</th>
    </tr>
    <tr>
      <th>Type</th>
      <th>cluster</th>
      <th></th>
      <th></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th rowspan="4" valign="top">H</th>
      <th>0</th>
      <td>0.010076</td>
      <td>397</td>
    </tr>
    <tr>
      <th>1</th>
      <td>0.006061</td>
      <td>165</td>
    </tr>
    <tr>
      <th>2</th>
      <td>0.027650</td>
      <td>217</td>
    </tr>
    <tr>
      <th>3</th>
      <td>0.044643</td>
      <td>224</td>
    </tr>
    <tr>
      <th rowspan="4" valign="top">L</th>
      <th>0</th>
      <td>0.024379</td>
      <td>2092</td>
    </tr>
    <tr>
      <th>1</th>
      <td>0.024738</td>
      <td>1051</td>
    </tr>
    <tr>
      <th>2</th>
      <td>0.042023</td>
      <td>1404</td>
    </tr>
    <tr>
      <th>3</th>
      <td>0.068135</td>
      <td>1453</td>
    </tr>
    <tr>
      <th rowspan="4" valign="top">M</th>
      <th>0</th>
      <td>0.016144</td>
      <td>1053</td>
    </tr>
    <tr>
      <th>1</th>
      <td>0.027668</td>
      <td>506</td>
    </tr>
    <tr>
      <th>2</th>
      <td>0.033422</td>
      <td>748</td>
    </tr>
    <tr>
      <th>3</th>
      <td>0.039130</td>
      <td>690</td>
    </tr>
  </tbody>
</table>
</div>

High-risk clusters: [3]
* Type L in high-risk clusters: 24.2%
* Type M in high-risk clusters: 23.0%
* Type H in high-risk clusters: 22.3%

### Typical Findings on this Dataset:

* Low-quality (L) machines often show higher representation in high-wear / high-risk clusters due to engineering reality (faster tool wear).
* If disproportionate without physical justification it is potential bias.

### Mitigation Strategies:

1. Stratified Analysis: Build separate cluster models per Type or include Type interactions.
2. Threshold Calibration: Use different risk thresholds/alerting rules per quality tier.
3. Fairness Constraints: During hyperparameter tuning, penalize solutions with high disparate impact.
4. Monitoring: Track cluster assignment parity in production and retrain if drift occurs.
5. Transparency: Document that Type is an engineering factor, not a protected class, but still ensure equitable maintenance outcomes.